# CI failure prediction on GitHub Actions - data collection

**Problem.** A CI run on `master` of a large project takes a long time and runs dozens of jobs. If we can predict, *before the run starts*, whether the run will fail, CI time can be prioritised (run likely failures first, notify early) or saved (skip likely passes).

**Research question.** Can a from-scratch, depth-limited CART decision tree, trained on pre-run GitHub Actions and git data, predict CI failures of a large OSS repository better than the history baselines (previous outcome, majority class)? Which pre-run signals drive its decisions?

The assignment requires collecting our own data. This notebook documents how it was collected and what the collected data looks like: coverage over time, run counts, label balance, failure streaks, and re-runs.

## Collection strategy

**Sources.** The targets in `config/targets.json`: repository + workflow pairs, each optionally restricted to one branch. The pooled dataset is analysed as if it came from one CI workflow; every pooled result is also reported per source (target slug). Runs of `push`, `pull_request` and `merge_group` carry a code change and are used. Runs of other events (schedule, dispatch) are listed but get no details.

### Pipeline (per target)

1. **Runs**: `GET /repos/<repo>/actions/workflows/<id>/runs?created=<window>`, plus `branch=` for a target restricted to one branch, one window per month. A window with >= 1000 runs (the API cap) is halved, on days and then on hours, until each part fits.
2. **First attempts**: the runs endpoint returns the *latest* attempt. For re-run runs we fetch `attempts/1`: the label is the first-attempt conclusion, which is what the developer saw first.
3. **Jobs**: fetched only for failed first attempts. They feed the previous run's failed-job features and the label check, which drops failures caused only by infrastructure jobs.
4. **Git**: one bare clone per repository, with every branch head and `refs/pull/*/head`, gives every diff, file history and author history without API calls.
5. **Table**: one chronological pass computes the features from runs **completed before** the current run was created. Previous-run features follow the run's lineage (its PR or branch).

Every response is cached on disk, so an interrupted collection resumes where it stopped. The API token is read from `gh auth token` at runtime and never written to disk. Author identities are replaced by salted hashes before any data is saved under `data/processed/`.

In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from acise_p2_cidt.analysis import labels
from acise_p2_cidt.collect.gh_runs import collect, read_json
from acise_p2_cidt.collect.targets import load_targets, slug
from acise_p2_cidt.data.build import build, write
from acise_p2_cidt.data.load import load, pool_name, schema_path, table_path
from acise_p2_cidt.viz import plots

# True: label check, every candidate on a 28-day sample.
# False: the real collection, the enabled targets over the window before C.
LABEL_CHECK = False
DEFERRED = {"openclaw__openclaw__209874334"}  # too large for the first pool; not sampled
# Collection date C: everything up to it is development data; runs after it are
# kept for the re-test of the frozen model. Pinned on the day of the full collection.
COLLECTION_UNTIL = date(2026, 11, 1)

if LABEL_CHECK:
    CONFIGS = [c for c in load_targets(enabled_only=False) if slug(c.target) not in DEFERRED]
    SINCE, UNTIL = date(2026, 10, 8), date(2026, 11, 4)
else:
    CONFIGS = load_targets()
    SINCE = (pd.Timestamp(COLLECTION_UNTIL) - pd.DateOffset(months=6)).date()
    UNTIL = COLLECTION_UNTIL
SOURCES = [slug(c.target) for c in CONFIGS]
DATASET = pool_name(SOURCES)
display(Markdown(f"**Sources** ({len(SOURCES)}; pooled dataset `{DATASET}`): " + ", ".join(f"`{s}`" for s in SOURCES)))

## 1. Collect runs

`LABEL_CHECK = True` collects the label-check sample: every candidate except the deferred ones, 2026-10-08 to 2026-11-04. `False` collects the enabled targets over the 6 months before C. Cached windows cost no calls, so the cell resumes after an interruption. Terminal equivalent: `uv run collect-runs --target <slug> --since <SINCE> --until <UNTIL>`.

In [ ]:
COLLECT_RETRIES: int = 10

manifests = {}

for c in CONFIGS:
    display(f"Source: {slug(c.target)}")
    
    for r in range(COLLECT_RETRIES):
        try:
            manifests[slug(c.target)] = read_json(collect(c.target, SINCE, UNTIL))
            break
        except Exception as e:
            display(f"[{r + 1} / {COLLECT_RETRIES}] ERROR: {str(e)}")
        
        if r == COLLECT_RETRIES - 1:
            raise RuntimeError(f"Could not retrieve info about: {slug(c.target)} in {COLLECT_RETRIES} retries!")

### Manifests

One row per source: runs, API calls, query windows, the largest window, and runs per event. Each window must stay under the API cap of **1000 runs**, otherwise its tail would be cut silently. The assert is the proof that no window was cut.

In [ ]:
manifest_table = (
    pd.DataFrame(
        {
            s: {
                "runs": m["n_runs"],
                "api_calls": m["api_calls"],
                "windows": len(m["windows"]),
                "largest_window": max(m["windows"].values(), default=0),
            }
            | {f"event: {k}": v for k, v in m["runs_per_event"].items()}
            for s, m in manifests.items()
        }
    )
    .T.fillna(0)
    .astype(int)
)
assert (manifest_table["largest_window"] < 1000).all(), manifest_table["largest_window"].idxmax()
display(Markdown('**Manifests per source**: runs collected, API calls, query windows, the largest window (must stay < 1000) and runs per event.'), manifest_table)

## 1.1 Label check

The label is the first-attempt conclusion. It is only useful if failures come from the code under test. For every registry entry, enabled or not, the check reads the cached runs (no API calls) and reports:

- **infra share**: failures whose failed jobs all match the target's `exclude_jobs`, plus failures without any failed job (the run failed before it tested the code). A pattern matches `<job>` or `<job> :: <failed step>`. These failures are dropped, not relabelled.
- **T2 failures**: failures whose previous run in the same lineage (PR or branch) passed, or that are the first run of their lineage. `est_t2_window` scales the sample count to the 6-month window (only in `LABEL_CHECK` mode).
- **flaky rate**: failed first attempts whose latest attempt passed.

Pass criteria: the workflow builds or tests the project on a supported event, infra share <= 30%, estimated T2 failures >= 20 for the pool and >= 50 for E1/E3.

In [ ]:
ALL_CONFIGS = load_targets(enabled_only=False)
summaries = {slug(c.target): labels.summary(c) for c in ALL_CONFIGS}
scale = 182.6 / ((UNTIL - SINCE).days + 1)  # window: (6 months) / sample days

label_table = pd.DataFrame(
    {
        slug(c.target): {
            "enabled": c.enabled,
            "exclude_jobs": len(c.exclude_jobs),
            "failed": s["n_failed"],
            "failure_rate": s["failure_rate"],
            "infra_share": s["infra_share"],
            "no_failed_job": s["n_no_failed_job"],
            "t2_failures": s["t2_failures"],
            "est_t2_window": round(s["t2_failures"] * scale) if LABEL_CHECK else None,
            "t2_share": s["t2_share"],
            "flaky_rate": s["flaky_rate"],
        }
        for c, s in zip(ALL_CONFIGS, summaries.values())
    }
).T
display(Markdown('**Label check per registry entry**: failures, infra share, T2 failures (with the 6-month estimate) and flaky rate.'), label_table)

In [ ]:
# Failed jobs left after exclusion, ranked until they cover >= 80% of the failures left
for s, x in summaries.items():
    if len(x["top_jobs"]):
        display(
            Markdown(f"**{s}**: failed jobs left after exclusion, until they cover >= 80% of the failures left"),
            x["top_jobs"],
        )

## 2. Build the per-run tables

One table per source. The build joins the source's API cache with its repository's git history. The first build of a repository clones it under `data/raw/` (every branch head and `refs/pull/*/head`) and fetches by SHA the run heads that a later push replaced. A table is built only if it does not exist yet: delete it to rebuild.

Kept: `success` and `failure` first attempts of `push`, `pull_request` and `merge_group` runs. Dropped and counted per reason: other conclusions, other events (`event_<name>`), and failures whose failed jobs are all infrastructure (`infra_failure`). `no_jobs_kept` counts failures kept although their jobs were not collected. `no_git` is the share of runs whose head and merge-base are both unreachable: their git features are NaN (<= 10% expected).

**Pool.** `load(SOURCES, ...)` concatenates the source tables and orders the runs by creation time, as if they came from one CI workflow. T2, the dev segment (the first 20% of the pooled runs) and the CV folds are taken on this pooled order. History features were computed inside each source before pooling, so a run's previous run is always from its own workflow and lineage.

In [ ]:
for c in CONFIGS:
    if not table_path(slug(c.target)).exists():
        display(f"Building: {slug(c.target)}")
        write(*build(c), c.target)

In [ ]:
schemas = {s: read_json(schema_path(s)) for s in SOURCES}
tables = {s: pd.read_csv(table_path(s), parse_dates=["created_at"]) for s in SOURCES}
# The pool needs the same columns in every source
mismatch = [s for s, sc in schemas.items() if sc["feature_cols"] != schemas[SOURCES[0]]["feature_cols"]]
assert not mismatch, mismatch

schema_table = pd.DataFrame(
    {
        s: {
            "runs": sc["n_runs"],
            "failed": sc["n_failed"],
            "fail_rate": round(sc["n_failed"] / sc["n_runs"], 3),
            "features": len(sc["feature_cols"]),
            "no_git": round(tables[s]["files_changed"].isna().mean(), 3),
            "first": tables[s]["created_at"].min().date(),
            "last": tables[s]["created_at"].max().date(),
        }
        | {f"dropped: {k}": v for k, v in sc["n_dropped"].items()}
        for s, sc in schemas.items()
    }
).T
schema_table = schema_table.fillna({c: 0 for c in schema_table if c.startswith("dropped: ")})

t1, t2 = load(SOURCES, "T1"), load(SOURCES, "T2")
pool_table = pd.DataFrame(
    {
        ds.meta["target"]: {
            "rows": ds.meta["n_rows"],
            "failed": ds.meta["n_failed"],
            "fail_rate": round(ds.y.mean(), 3),
            "dev_rows": ds.meta["n_dev"],
            "dev_end": ds.order.iloc[ds.meta["n_dev"]].date(),
            "features": ds.X.shape[1],
        }
        for ds in (t1, t2)
    }
)
display(
    Markdown("**Built tables per source**: runs kept, failure rate, share of runs without git features (`no_git`), period, and runs dropped per reason."),
    schema_table,
    Markdown(f"**Pooled dataset `{DATASET}`** ({len(SOURCES)} sources), T1 and T2: rows, failures, the dev segment (first 20% of the pooled rows) and the feature count."),
    pool_table,
)

## 3. Coverage over time

Runs and failure rate per month, per source and pooled (`all`). Gaps would mean missing windows; a moving failure rate means drift that the time-ordered evaluation has to cope with.

In [ ]:
runs = pd.concat([t.assign(source=s) for s, t in tables.items()], ignore_index=True)
# Per-source rows plus the pool as source "all". Lineage names are unique only
# within a source, so the pooled copy prefixes them with the source.
both = pd.concat(
    [runs, runs.assign(lineage=runs["source"] + "/" + runs["lineage"], source="all")],
    ignore_index=True,
)
both["source"] = pd.Categorical(both["source"], categories=[*SOURCES, "all"])
month = both["created_at"].dt.strftime("%Y-%m").rename("month")
monthly = both.groupby(["source", month], observed=True)["failed"].agg(runs="size", fail_rate="mean")
display(
    Markdown("**Runs per month**, per source and pooled (`all`)"),
    monthly["runs"].unstack(fill_value=0),
    Markdown("**Failure rate per month**, per source and pooled (`all`)"),
    monthly["fail_rate"].unstack().round(3),
)

**Failure-rate shifts.** A source's failure rate can jump for weeks for reasons outside the code (a failing CI leg, a broken runner image). The table lists the months where a source's rate is above twice, or below half, its median monthly rate.

The first plot shows the weekly failure rate, one line per source plus the pool (`all`). The second shows the pooled runs and failure rate per week (T1); the dashed line ends the dev segment, the first 20% of the pooled runs, which is used for EDA and feature selection only.

In [ ]:
weekly = (
    both.groupby(["source", pd.Grouper(key="created_at", freq="W")], observed=True)["failed"]
    .agg(runs="size", fail_rate="mean")
    .reset_index()
)
fig = px.line(
    weekly, x="created_at", y="fail_rate", color="source", hover_data=["runs"],
    title="Weekly failure rate per source and pooled (all)", labels={"created_at": "week", "fail_rate": "failure rate"},
)
display(plots.save(fig.update_layout(yaxis_tickformat=".0%"), "collection_fail_rate_per_source"))

fig = plots.runs_per_week(runs).add_vline(x=t1.order.iloc[t1.meta["n_dev"]], line_dash="dash")
fig.update_layout(
    title="Runs and failure rate per week, pooled T1 (dashed: end of the dev segment)",
    yaxis2_tickformat=".0%", legend={"orientation": "h", "y": 1.1},
)
display(plots.save(fig, "collection_runs_per_week_pool"))

In [ ]:
src_monthly = monthly.query("source != 'all'")
median = src_monthly["fail_rate"].groupby(level="source", observed=True).transform("median")
shifts = src_monthly.assign(median_rate=median)[
    (src_monthly["fail_rate"] > 2 * median) | (src_monthly["fail_rate"] < median / 2)
].round(3)
display(Markdown("**Failure-rate shifts per source**: months whose rate is above 2x or below 1/2 of the source's median monthly rate."), shifts)

### Events and lineages

A **lineage** is the PR or branch a run belongs to: previous-run features and T2 follow it. PR runs are keyed by their head repository and branch, merge-queue runs by their PR, pushes by their branch. The first run of a lineage has no known previous run (`lineage_first_run`).

In [ ]:
def bucket_counts(df: pd.DataFrame, col: str, title: str):
    """Bars per source: how many rows of `df` fall in each power-of-2 bucket of `col`
    (1, 2, 3-4, 5-8, ...), count on a log axis. Most values are 1 or 2, a few reach hundreds."""
    k = np.ceil(np.log2(df[col])).astype(int)
    counts = df.assign(k=k).groupby(["source", "k"], observed=True).size().rename("count").reset_index()
    label = lambda k: str(2**k) if k < 2 else f"{2 ** (k - 1) + 1}-{2**k}"
    counts[col] = counts["k"].map(label)
    return (
        px.bar(
            counts, x=col, y="count", facet_col="source", facet_col_wrap=3, facet_col_spacing=0.06, log_y=True,
            category_orders={col: [label(i) for i in range(counts["k"].max() + 1)]},
            title=title, height=260 * ((counts["source"].nunique() + 2) // 3) + 100,
        )
        .for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font_size=11))
        .update_xaxes(showticklabels=True)
        .update_yaxes(matches=None, showticklabels=True)
    )


by_event = both.groupby(["source", "event"], observed=True)["failed"].agg(runs="size", fail_rate="mean")
display(Markdown("**Runs and failure rate per event**, per source and pooled (`all`)"), by_event.unstack("event").round(3))

per_lineage = both.groupby(["source", "lineage"], observed=True).size().rename("runs")
by_src = both.groupby("source", observed=True)
lineage_table = pd.DataFrame(
    {
        "lineages": per_lineage.groupby(level="source", observed=True).size(),
        "median_runs": per_lineage.groupby(level="source", observed=True).median(),
        "max_runs": per_lineage.groupby(level="source", observed=True).max(),
        "first_of_lineage": by_src["lineage_first_run"].mean(),
        "fork_share": by_src["from_fork"].mean(),
    }
).round(3)
display(
    Markdown("**Lineages per source and pooled (`all`)**: count, runs per lineage, share of runs that open a "
             "lineage, share of runs from forks."),
    lineage_table,
)
plots.save(
    bucket_counts(per_lineage.reset_index(), "runs", "Lineages by number of runs, per source and pooled (log count)"),
    "collection_runs_per_lineage",
)

## 4. Label balance - T1 vs. T2

- **T1**: all runs.
- **T2**: runs whose previous completed run in the same lineage passed, or that are the first run of their lineage (*first* failures). A failure that follows a failure of the same PR or branch is easy to guess (Parrot baseline); T2 removes those and keeps the harder case.

A source joins the pool with >= 20 T2 failures, and is analysed on its own (E1, E3) with >= 50. The pooled row (`all`) is the dataset the models are trained on.

In [ ]:
by_source = {
    name: {**{s: load(s, name) for s in SOURCES}, "all": pooled}
    for name, pooled in (("T1", t1), ("T2", t2))
}
balance = pd.concat(
    [
        pd.DataFrame({"failed": ds.y, "target": name, "source": s})
        for name, data in by_source.items()
        for s, ds in data.items()
    ]
)
balance["source"] = pd.Categorical(balance["source"], categories=[*SOURCES, "all"])
counts = balance.groupby(["source", "target"], observed=True)["failed"].agg(runs="size", failed="sum", fail_rate="mean")
balance_table = counts.unstack("target").round(3)
is_source = balance_table.index != "all"  # eligibility belongs to a source, not to the pool
balance_table[("eligible", "pool")] = (balance_table[("failed", "T2")] >= 20).where(is_source)
balance_table[("eligible", "e1_e3")] = (balance_table[("failed", "T2")] >= 50).where(is_source)
display(
    Markdown("**Label balance, T1 vs. T2**, per source and pooled (`all`): runs, failures, failure rate, and whether "
             "the source joins the pool (>= 20 T2 failures) and E1/E3 (>= 50)."),
    balance_table,
)
counts = counts.reset_index()
fig = px.bar(
    counts, y="source", x="failed", color="target", barmode="group", orientation="h", log_x=True,
    text=counts["fail_rate"].map("{:.0%}".format), hover_data=["runs"], height=60 * len(SOURCES) + 210,
    title="Failures per source and pooled (all), T1 vs. T2 (log count; bar text: failure rate;<br>"
    "dotted: pool needs 20 T2 failures, dashed: E1/E3 need 50)",
    labels={"failed": "failed runs"},
)
fig.update_traces(textposition="outside", cliponaxis=False).update_yaxes(autorange="reversed")
fig.add_vline(x=20, line_dash="dot").add_vline(x=50, line_dash="dash")
plots.save(fig, "collection_label_balance")

## 5. Failure streaks

Length of consecutive-failure streaks **within a lineage**, in creation order (log scale): the runs of different PRs interleave, so a streak over the whole workflow would mix unrelated changes. The pooled row (`all`) collects the streaks of every source. Long streaks mean a broken branch stays broken over several pushes. That makes the previous outcome a strong baseline on T1, and is the reason T2 exists.

In [ ]:
def fail_streaks(df: pd.DataFrame) -> pd.Series:
    """Lengths of the failure streaks of every lineage."""
    out = []
    for _, s in df.sort_values("created_at", kind="stable").groupby("lineage")["failed"]:
        g = s.groupby((s != s.shift()).cumsum()).agg(["first", "size"])
        out.append(g.loc[g["first"] == 1, "size"])
    return pd.concat(out, ignore_index=True) if out else pd.Series(dtype=int)


groups = dict(list(both.groupby("source", observed=True)))
streaks = {s: fail_streaks(t) for s, t in groups.items()}
streak_table = pd.DataFrame(
    {
        s: {
            "streaks": len(st),
            "median": st.median(),
            "mean": st.mean(),
            "max": st.max(),
            "follow_a_failure": (st - 1).sum() / groups[s]["failed"].sum(),
        }
        for s, st in streaks.items()
    }
).T.round(3)
display(
    Markdown("**Failure streaks within a lineage**, per source and pooled (`all`): count, length, and the share of "
             "failures that follow a failure of the same lineage."),
    streak_table,
)
lengths = pd.concat([st.rename("length").to_frame().assign(source=s) for s, st in streaks.items()])
plots.save(
    bucket_counts(lengths, "length", "Failure streaks by length, within a lineage, per source and pooled (log count)"),
    "collection_fail_streaks",
)

## 6. Re-runs and flaky share

A run is **flaky** when its first attempt failed and a later attempt of the same run passed. Nothing in the code changed, so such failures cannot be predicted from the change itself.

Teams that fix a failure with a new commit rarely re-run, so where re-runs are rare the flaky share is a **lower bound** of the label noise. `was_rerun` and `flaky` are diagnostics only and never used as features (they are known only after the run is completed).

In [ ]:
failed = both[both["failed"] == 1]
flaky_table = pd.DataFrame(
    {
        "rerun_share": both.groupby("source", observed=True)["was_rerun"].mean(),
        "failures_rerun": failed.groupby("source", observed=True)["was_rerun"].mean(),
        "flaky_share_of_failures": failed.groupby("source", observed=True)["flaky"].mean(),
    }
).round(4)
first = both["failed"].map({0: "success", 1: "failure"}).rename("first attempt")
display(
    Markdown("**First vs. latest attempt**, per source and pooled (`all`): a `failure` row with latest `success` is flaky."),
    pd.crosstab([both["source"], first], both["conclusion_latest"].rename("latest attempt")),
    Markdown("**Re-runs and flaky share**, per source and pooled (`all`)"),
    flaky_table,
)

## Observations

_To be written after the run on the collected data._